In [42]:
import os
from pathlib import Path

# Keep model cache off the system drive.
os.environ["HF_HOME"] = r"E:\devtools\huggingface"

import numpy as np
import polars as pl

from sentence_transformers import SentenceTransformer

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data/raw/dataset"

query = pl.read_csv(
    DATA_DIR / "query.csv",
    separator="\t",
)

texts = query["query"].to_list()
query_ids = query["query_id"].to_numpy()

In [43]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

embeddings = model.encode(texts,
                          batch_size=64,
                          show_progress_bar=True,
                          normalize_embeddings=True,
                          )

print(f"Embeddings shape: {embeddings.shape}")

# L2 нормы проверка
assert np.allclose(
    np.linalg.norm(embeddings, axis=1),
    1.0,
    atol=1e-6,
)

Batches: 100%|██████████| 8/8 [00:00<00:00, 17.54it/s]

Embeddings shape: (480, 384)


будем сравнивать по косинусной близости

$$cos(a,b)=\frac{a^Tb}{||a|| ||b||}=a^Tb$$

In [44]:
# матрица сходства
semantic_similarity = embeddings @ embeddings.T
print(np.triu(semantic_similarity, k=1))

[[0.         0.23247555 0.21225666 ... 0.13163391 0.4020673  0.20359673]
 [0.         0.         0.05417186 ... 0.17547446 0.15728915 0.3047381 ]
 [0.         0.         0.         ... 0.1328338  0.12169845 0.14920522]
 ...
 [0.         0.         0.         ... 0.         0.0739954  0.15530902]
 [0.         0.         0.         ... 0.         0.         0.1819424 ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


In [45]:
# Вывести топ-100 похожих пар с query_id
i, j = np.triu_indices_from(semantic_similarity, k=1)
semantic_scores = semantic_similarity[i, j]
top = np.argsort(-semantic_scores)[:100]

query_ids = query["query_id"].to_numpy()

top_100 = pl.DataFrame({
    "query_id_1": query_ids[i[top]],
    "query_id_2": query_ids[j[top]],
    "query_1": [texts[idx] for idx in i[top]],
    "query_2": [texts[idx] for idx in j[top]],
    "semantic_similarity": semantic_scores[top],
})


In [46]:
# добавил TF-IDF вариант

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    lowercase=True,
)

vectors = vectorizer.fit_transform(texts)
lexical_similarity = cosine_similarity(vectors)


i, j = np.triu_indices_from(lexical_similarity, k=1)
lexical_scores = lexical_similarity[i, j]

top_100 = top_100.with_columns(
    lexical_similarity=pl.Series("lexical_similarity", lexical_scores[top])
)

with pl.Config(tbl_rows=100):
    display(top_100)

query_id_1,query_id_2,query_1,query_2,semantic_similarity,lexical_similarity
i64,i64,str,str,f32,f64
118,314,"""body pillow case""","""body pillow and case""",0.955746,0.939891
32,429,"""dark gray dresser""","""gray dresser""",0.947676,0.81286
26,126,"""leather chairs""","""leather dining chairs""",0.9315,0.814871
81,137,"""orren ellis l shape desk""","""orren ellis l shaped desk""",0.929075,0.917201
26,151,"""leather chairs""","""leather chair""",0.905433,0.863574
151,242,"""leather chair""","""accent leather chair""",0.8816,0.764915
103,231,"""kitchen anti fatigue mats""","""anti fatigue mat""",0.863668,0.771943
177,428,"""rustic storage cabinet""","""kitchen storage cabinet""",0.862831,0.621583
151,478,"""leather chair""","""worn leather office chair""",0.85813,0.613822


In [47]:
# без semantic similarity не обойтись, так как TF-IDF
# пропускает такие примеры:
# "wall shelves"	"wall design shelf"

# Мини-датасет для ручного анализа

In [49]:
i, j = np.triu_indices_from(semantic_similarity, k=1)

semantic_scores = semantic_similarity[i, j]
lexical_scores = lexical_similarity[i, j]

semantic_top = np.argsort(-semantic_scores)[:100]
lexical_top = np.argsort(-lexical_scores)[:100]

candidates = np.union1d(semantic_top, lexical_top)

candidate_pairs = pl.DataFrame({
    "query_id_a": query_ids[i[candidates]],
    "query_id_b": query_ids[j[candidates]],
    "query_a": [texts[idx] for idx in i[candidates]],
    "query_b": [texts[idx] for idx in j[candidates]],
    "semantic_similarity": semantic_scores[candidates],
    "lexical_similarity": lexical_scores[candidates],
})

print(candidate_pairs.height)

150


In [51]:
candidate_pairs.write_csv(ROOT_DIR / "data" / "annots" / "candidate_pairs.csv", separator="\t")